# How Small Language Models Work
## Hands-On with Qwen & OLMo (< 5B Parameters)

Small language models (under 5 billion parameters) are practical, efficient, and can run on consumer hardware. In this notebook you will:

1. **Understand** what makes a model "small" and why it matters
2. **Download** quantized models from Hugging Face using `llama-cpp-python`
3. **Run inference** with Qwen and OLMo models locally
4. **Compare** their outputs and characteristics
5. **Practice** prompting, generation tuning, and evaluation

### Prerequisites
- Python 3.10+
- ~4 GB free disk space (for quantized model files)
- A machine with at least 8 GB RAM (no GPU required for quantized models)

---

## 1. What Are Small Language Models?

Language models are neural networks trained to predict text. The "size" refers to the number of **parameters** (learned weights).

| Category | Parameters | Examples | Typical Hardware |
|----------|-----------|----------|------------------|
| Tiny | < 500M | DistilGPT-2, TinyLlama | CPU, mobile |
| Small | 500M – 5B | **Qwen2.5-1.5B, OLMo-1B** | Laptop CPU/GPU |
| Medium | 5B – 30B | Llama-3-8B, Mistral-7B | Desktop GPU |
| Large | 30B+ | Llama-3-70B, GPT-4 | Multi-GPU / Cloud |

### Why Small Models?
- **Fast inference** — generate tokens in milliseconds on a laptop
- **Low cost** — no cloud GPU needed
- **Privacy** — data never leaves your machine
- **Quantization** — GGUF format compresses weights (e.g., Q4_K_M ≈ 4-bit) so a 1.5B model fits in ~1 GB RAM

### How They Work (Simplified)
1. **Tokenization**: Text → integer tokens
2. **Embedding**: Tokens → dense vectors
3. **Transformer layers**: Self-attention + feed-forward networks process context
4. **Output head**: Predicts probability distribution over next token
5. **Sampling**: Pick next token (greedy, top-k, top-p, temperature)

Smaller models have fewer transformer layers and smaller hidden dimensions, trading some capability for speed.

## 2. Environment Setup

We install `llama-cpp-python` which provides:
- A Python binding to **llama.cpp** (C++ inference engine for GGUF models)
- Automatic downloading from Hugging Face Hub
- Efficient CPU inference with quantized weights

We also install `huggingface-hub` for browsing and downloading model files.

In [ ]:
# Install dependencies (run once)
!pip install llama-cpp-python huggingface-hub --quiet

In [ ]:
import os
import time
from llama_cpp import Llama
from huggingface_hub import hf_hub_download

print("Setup complete!")

## 3. Qwen2.5-1.5B — Download & Run

[Qwen2.5](https://huggingface.co/Qwen) is a family of models from Alibaba Cloud. The **1.5B** variant is an excellent small model:
- 1.5 billion parameters
- Trained on multilingual data (English, Chinese, code, math)
- Strong reasoning for its size
- GGUF quantized versions available on Hugging Face

We will download a **Q4_K_M** quantization (~1 GB) which balances quality and speed.


In [ ]:
# Download Qwen2.5-1.5B (GGUF Q4_K_M quantization, ~1 GB)
qwen_model_path = hf_hub_download(
    repo_id="Qwen/Qwen2.5-1.5B-Instruct-GGUF",
    filename="qwen2.5-1.5b-instruct-q4_k_m.gguf",
)
print(f"Model downloaded to: {qwen_model_path}")
print(f"File size: {os.path.getsize(qwen_model_path) / 1e9:.2f} GB")


In [ ]:
# Load Qwen2.5-1.5B into memory
qwen = Llama(
    model_path=qwen_model_path,
    n_ctx=2048,       # context window size (tokens)
    n_threads=4,      # CPU threads for inference
    verbose=False,
)
print(f"Model loaded! Context window: {qwen.n_ctx()} tokens")


In [ ]:
# Generate text with Qwen2.5-1.5B
prompt = "Explain what a neural network is in 3 sentences."

start = time.time()
response = qwen.create_chat_completion(
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ],
    max_tokens=256,
    temperature=0.7,
    top_p=0.9,
)
elapsed = time.time() - start

answer = response["choices"][0]["message"]["content"]
tokens_used = response["usage"]["completion_tokens"]

print(f"Prompt: {prompt}")
print(f"\nQwen2.5-1.5B Response:\n{answer}")
print(f"\nTokens generated: {tokens_used}")
print(f"Time: {elapsed:.2f}s ({tokens_used/elapsed:.1f} tokens/sec)")


### 🎯 Practice 1: Explore Generation Parameters

Try modifying the cell above with different settings and re-run it:

1. **Temperature**: Change `temperature=0.7` to `0.1` (more deterministic) or `1.5` (more creative). What happens?
2. **Max tokens**: Set `max_tokens=50`. Does the answer still make sense when cut short?
3. **Prompt engineering**: Try these prompts:
   - `"Write a Python function that sorts a list"` (code generation)
   - `"Translate 'Hello, how are you?' to French and Spanish"` (multilingual)
   - `"What is 247 * 13?"` (arithmetic — where do small models struggle?)


In [ ]:
# 🎯 Practice 1: Your turn! Modify the prompt and parameters below, then run this cell.
my_prompt = "Write a Python function that sorts a list"  # <-- Change this!
my_temperature = 0.7   # Try: 0.1, 0.7, 1.5
my_max_tokens = 256    # Try: 50, 128, 512

response = qwen.create_chat_completion(
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": my_prompt},
    ],
    max_tokens=my_max_tokens,
    temperature=my_temperature,
    top_p=0.9,
)
print(response["choices"][0]["message"]["content"])


## 4. OLMo-1B — Download & Run

[OLMo](https://allenai.org/olmo) (Open Language Model) is from the Allen Institute for AI (AI2). What makes OLMo special:
- **Fully open**: Weights, data, training code, and evaluation all released
- **1B parameters** — even smaller than Qwen2.5-1.5B
- **Dolma dataset** — trained on a curated, documented open dataset
- Great for research and understanding model internals

We will download OLMo-1B in GGUF Q4_K_M format.


In [ ]:
# Download OLMo-1B (GGUF Q4_K_M quantization, ~0.7 GB)
olmo_model_path = hf_hub_download(
    repo_id="allenai/OLMo-1B-0724-hf-GGUF",
    filename="olmo-1b-0724-hf-q4_k_m.gguf",
)
print(f"Model downloaded to: {olmo_model_path}")
print(f"File size: {os.path.getsize(olmo_model_path) / 1e9:.2f} GB")


In [ ]:
# Load OLMo-1B into memory
olmo = Llama(
    model_path=olmo_model_path,
    n_ctx=2048,
    n_threads=4,
    verbose=False,
)
print(f"OLMo loaded! Context window: {olmo.n_ctx()} tokens")


In [ ]:
# Generate text with OLMo-1B
# Note: OLMo-1B is a base (completion) model, not instruction-tuned,
# so we use text completion rather than chat format.
prompt = "The key difference between a CPU and a GPU is that"

start = time.time()
response = olmo(
    prompt,
    max_tokens=256,
    temperature=0.7,
    top_p=0.9,
    echo=False,
)
elapsed = time.time() - start

answer = response["choices"][0]["text"]
tokens_used = response["usage"]["completion_tokens"]

print(f"Prompt: {prompt}")
print(f"\nOLMo-1B Completion:\n{answer}")
print(f"\nTokens generated: {tokens_used}")
print(f"Time: {elapsed:.2f}s ({tokens_used/elapsed:.1f} tokens/sec)")


### 🎯 Practice 2: Base Model vs. Instruct Model

OLMo-1B is a **base** model — it was trained to predict next tokens, not to follow instructions. Qwen2.5-1.5B-Instruct was **fine-tuned** to follow instructions.

Try these experiments:
1. Give OLMo the same prompt style as Qwen: `"Explain what a neural network is in 3 sentences."` — Does it follow the instruction, or just continue the text?
2. Give OLMo a **completion-style** prompt: `"Neural networks are inspired by"` — Is this better?
3. What does this tell you about the difference between **base** and **instruct** models?


In [ ]:
# 🎯 Practice 2: Your turn! Test OLMo with different prompt styles.
my_olmo_prompt = "Neural networks are inspired by"  # <-- Change this!

response = olmo(
    my_olmo_prompt,
    max_tokens=200,
    temperature=0.7,
)
print(f"Prompt: {my_olmo_prompt}")
print(f"Completion: {response['choices'][0]['text']}")


## 5. Head-to-Head Comparison

Let's run both models on the same prompts to see how size, training data, and fine-tuning affect output quality.


In [ ]:
# Compare both models on the same prompts
test_prompts = [
    "The capital of France is",
    "def fibonacci(n):\n",
    "Machine learning is different from traditional programming because",
]

for prompt in test_prompts:
    print("=" * 60)
    print(f"PROMPT: {prompt!r}\n")

    # Qwen (instruct model, but using completion mode for fair comparison)
    qwen_out = qwen(prompt, max_tokens=100, temperature=0.3, echo=False)
    print(f"QWEN 2.5-1.5B:\n{qwen_out['choices'][0]['text'].strip()}\n")

    # OLMo (base model)
    olmo_out = olmo(prompt, max_tokens=100, temperature=0.3, echo=False)
    print(f"OLMo-1B:\n{olmo_out['choices'][0]['text'].strip()}\n")


### 🎯 Practice 3: Token Probabilities & Sampling

Understanding how models pick the next token is key to understanding LLMs. Let's peek under the hood.

The `logprobs` parameter makes the model return the log-probabilities of the top candidate tokens at each step. This shows you what the model "almost" said.

In [ ]:
# Explore token probabilities
import math

prompt = "The meaning of life is"

response = qwen(
    prompt,
    max_tokens=20,
    temperature=0.0,  # greedy - always pick highest probability token
    logprobs=5,       # return top-5 token candidates at each step
    echo=False,
)

print(f"Prompt: {prompt!r}\n")
print(f"Completion: {response['choices'][0]['text']}\n")
print("Token-by-token breakdown (top 5 candidates):")
print("-" * 50)

for i, logprob_entry in enumerate(response["choices"][0]["logprobs"]["top_logprobs"]):
    token = response["choices"][0]["logprobs"]["tokens"][i]
    print(f"\nPosition {i}: chose {token!r}")
    for tok, lp in sorted(logprob_entry.items(), key=lambda x: x[1], reverse=True):
        prob = math.exp(lp) * 100
        bar = "#" * int(prob / 2)
        print(f"  {tok!r:20s} {prob:5.1f}% {bar}")

### 🎯 Practice 4: Build a Simple Evaluation

A critical skill: how do you **measure** model quality? Write a mini-evaluation that tests both models on factual questions.

Fill in the `test_cases` list below with question-answer pairs, then run the cell to see how each model scores.

In [ ]:
# Practice 4: Mini-evaluation framework
# Add your own test cases! Format: (prompt_prefix, expected_substring)
test_cases = [
    ("The capital of Japan is", "Tokyo"),
    ("Water freezes at", "0"),
    ("The Python keyword for defining a function is", "def"),
    # Add more below:
    # ("The largest planet in our solar system is", "Jupiter"),
]

print("Model Evaluation Results")
print("=" * 60)

for model_name, model in [("Qwen2.5-1.5B", qwen), ("OLMo-1B", olmo)]:
    correct = 0
    print(f"\n{model_name}:")
    for prompt, expected in test_cases:
        out = model(prompt, max_tokens=30, temperature=0.0, echo=False)
        completion = out["choices"][0]["text"].strip()
        passed = expected.lower() in completion.lower()
        correct += int(passed)
        status = "PASS" if passed else "FAIL"
        print(f"  {status} {prompt!r}")
        if not passed:
            print(f"      Expected: {expected}, got: {completion[:60]!r}")
    print(f"  Score: {correct}/{len(test_cases)}")

## 6. Key Takeaways & Next Steps

### What We Learned
- **Small models (< 5B)** can run on a laptop CPU using GGUF quantization
- **llama-cpp-python** makes it easy to download and run models from Hugging Face
- **Instruct vs. Base**: Qwen2.5 (instruct-tuned) follows instructions; OLMo (base) is better at text completion
- **Quantization** (Q4_K_M) shrinks models ~4x with minimal quality loss
- **Sampling parameters** (temperature, top_p, max_tokens) control output style

### When to Use Small Models
| Use Case | Good Fit? | Why |
|----------|-----------|-----|
| Text classification | Yes | Simple task, fast inference |
| Code completion | Decent | Qwen handles basic code well |
| Long-form writing | Limited | May lose coherence after a few paragraphs |
| Complex reasoning | Poor | Need 7B+ for multi-step logic |
| Summarization | Decent | Works for short documents |

### Next Steps
1. Try **Qwen2.5-3B** for a step up in quality (still fits in 2-3 GB)
2. Explore **GPU acceleration** with `llama-cpp-python` CUDA/Metal builds
3. Fine-tune a small model on your own data using QLoRA
4. Compare with other small models: TinyLlama, Phi-3-mini, Gemma-2B

### 🎯 Practice 5: Challenge Exercises

1. **Benchmark**: Time both models generating 100 tokens and calculate tokens/second. Which is faster? Why?
2. **System prompts**: Does changing the system prompt for Qwen change the style of responses? Try: "You are a pirate" vs "You are a scientist"
3. **Context window**: What happens if you pass a very long prompt (>1000 tokens)? Try feeding in a long paragraph and asking for a summary.
4. **Model card reading**: Visit the Hugging Face model cards for [Qwen2.5-1.5B](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct-GGUF) and [OLMo-1B](https://huggingface.co/allenai/OLMo-1B-0724-hf-GGUF). What training data was used? What benchmarks are reported?

In [ ]:
# Practice 5: Scratch space for challenge exercises
# Use this cell for your experiments!

# Example: Benchmark token generation speed
import time

prompt = "Once upon a time"
for name, model in [("Qwen2.5-1.5B", qwen), ("OLMo-1B", olmo)]:
    start = time.time()
    out = model(prompt, max_tokens=100, temperature=0.7)
    elapsed = time.time() - start
    toks = out["usage"]["completion_tokens"]
    print(f"{name}: {toks} tokens in {elapsed:.2f}s = {toks/elapsed:.1f} tok/s")

## Cleanup

Free the model memory when you are done.

In [ ]:
# Free model memory
del qwen
del olmo
print("Models unloaded.")